# 02 - StackOverflow Preprocessing

**Goal**: Clean, filter, and format StackOverflow data for labeling.

**Steps**:
1. Load `stackoverflow_full_data.csv`.
2. Remove duplicates (by ID).
3. Standardize columns (`source_type`, `raw_text`, `tech_keywords`, `event_date`).
4. Save to `stackoverflow_preprocessed.csv`.


XÓA CỘT ID, ĐƯA TIME VỀ ĐÚNG DẠNG

In [1]:
import pandas as pd
import os

# Paths
INPUT_FILE = 'data/stackoverflow_full_data.csv'
OUTPUT_FILE = 'data/stackoverflow_preprocessed.csv'

# Check if input exists in current dir, or check root if running from notebooks dir
if not os.path.exists(INPUT_FILE) and os.path.exists(os.path.join('..', INPUT_FILE)):
    INPUT_FILE = os.path.join('..', INPUT_FILE)
    OUTPUT_FILE = os.path.join('..', OUTPUT_FILE)

print(f"Reading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
print(f"Initial rows: {len(df)}")

Reading data/stackoverflow_full_data.csv...
Initial rows: 7672347


In [2]:
# 1. Remove Duplicates
df.drop_duplicates(subset=['id'], keep='first', inplace=True)
print(f"Rows after deduplication: {len(df)}")

# 2. Filter (Optional - user mentioned 'filter', assuming basic validity checks)
# Ensure title and tags are not null
df = df.dropna(subset=['title', 'tags'])
print(f"Rows after dropna: {len(df)}")

Rows after deduplication: 7672347
Rows after dropna: 7672343


In [3]:
# 3. Standardize Schema
# Requirements: source_type, raw_text, tech_keywords, event_date
# All text lowercase

# Filter by year (2023-2025)
print(f"Rows before date filtering: {len(df)}")
df['creation_date_dt'] = pd.to_datetime(df['creation_date'], format='ISO8601', errors='coerce')
df = df[df['creation_date_dt'].dt.year.between(2023, 2025)]
print(f"Rows after filtering (2023-2025): {len(df)}")

df['source_type'] = 'stackoverflow'
df['raw_text'] = df['title'].astype(str).str.lower()
df['tech_keywords'] = df['tags'].astype(str).str.lower().str.replace('><', ', ').str.replace('<', '').str.replace('>', '')

# Define synonymous tags (canonical -> list of variants)
tag_synonyms = {
    # --- Languages ---
    'python': ['python-3.x', 'python-2.x', 'python3', 'python2', 'py'],
    'javascript': ['js', 'ecmascript', 'es6', 'es5'],
    'typescript': ['ts'],
    'java': ['jdk', 'jre'],
    'c#': ['csharp', '.net core', 'dotnet'],
    'c++': ['cpp', 'c++11', 'c++14', 'c++17', 'c++20', 'c++23'],
    'go': ['golang'],
    'rust': ['rust-lang'],
    'php': ['php7', 'php8'],
    'ruby': ['ror', 'ruby-on-rails'],
    'swift': ['swiftui'],
    'kotlin': ['kotlin-android'],
    'scala': [],
    'r': ['r-lang'],
    'dart': [],
    'shell': ['bash', 'zsh', 'sh', 'linux-shell'],
    'sql': ['mysql', 'postgresql', 'postgres', 'plsql', 'tsql'], 
    
    # --- Databases ---
    'postgresql': ['postgres', 'pgsql'],
    'mysql': ['mariadb'],
    'sql-server': ['mssql', 't-sql', 'tsql'],
    'mongodb': ['mongo', 'mongoose'],
    'redis': [],
    'elasticsearch': ['elk', 'kibana', 'logstash'],
    
    # --- Frontend ---
    'reactjs': ['react', 'react.js', 'react-native'], # Aggressive grouping. 
    'angular': ['angularjs', 'angular.js', 'ng', 'angular2+', 'angular2'], 
    'vue.js': ['vue', 'vuejs', 'vue-js', 'vue2', 'vue3'],
    'next.js': ['nextjs', 'next-js'],
    'node.js': ['nodejs', 'node', 'express', 'express.js'], # Express often implies Node.
    
    # --- Backend/Frameworks ---
    'spring': ['spring-boot', 'springboot', 'spring-mvc'],
    'django': ['django-rest-framework', 'drf'],
    'flask': [],
    'fastapi': [],
    'laravel': [],
    '.net': ['asp.net', 'asp.net-core', 'entity-framework'],
    'ruby-on-rails': ['rails'],
    
    # --- Cloud/DevOps ---
    'aws': ['amazon-web-services', 'ec2', 's3', 'lambda'],
    'azure': ['azure-devops'],
    'gcp': ['google-cloud', 'google-cloud-platform'],
    'docker': ['docker-compose'],
    'kubernetes': ['k8s', 'helm'],
    'terraform': ['tf'],
    'jenkins': [],
    'git': ['github', 'gitlab', 'bitbucket'],
    
    # --- AI/ML ---
    'tensorflow': ['tf'],
    'pytorch': ['torch'],
    'pandas': [],
    'numpy': [],
    'scikit-learn': ['sklearn'],
    'opencv': ['cv2'],
    'natural-language-processing': ['nlp'],
    'computer-vision': ['cv'],
    'machine-learning': ['ml'],
    'deep-learning': ['dl']
}

# Flatten to map for O(1) lookup
normalization_map = {variant: canonical for canonical, variants in tag_synonyms.items() for variant in variants}

def normalize_tags(tag_string):
    if not isinstance(tag_string, str):
        return ''
    # Split by comma
    tags = [t.strip() for t in tag_string.split(',')]
    # Normalize
    normalized = [normalization_map.get(t, t) for t in tags]
    # Deduplicate while preserving order
    unique_tags = list(dict.fromkeys(normalized))
    return ', '.join(unique_tags)

df['tech_keywords'] = df['tech_keywords'].apply(normalize_tags)

df['event_date'] = df['creation_date_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S.%f%z')

# Select Final Columns
final_cols = ['id', 'source_type', 'raw_text', 'tech_keywords', 'event_date']
df_clean = df[final_cols].copy()

print(df_clean.head())

Rows before date filtering: 7672343
Rows after filtering (2023-2025): 391836
               id    source_type  \
7280511  75608323  stackoverflow   
7280512  75602063  stackoverflow   
7280513  75574268  stackoverflow   
7280514  75263047  stackoverflow   
7280515  75078744  stackoverflow   

                                                  raw_text  \
7280511  how do i solve "error: externally-managed-envi...   
7280512  pip install -r requirements.txt is failing: "t...   
7280513    missing file libarclite_iphoneos.a (xcode 14.3)   
7280514                  duplicate class in kotlin android   
7280515  docker: error response from daemon: failed to ...   

                                    tech_keywords  \
7280511  python, pip, debian, failed-installation   
7280512                             python, linux   
7280513                       ios, xcode, xcode14   
7280514       java, android, kotlin, dependencies   
7280515                                docker, go   

              

In [4]:
# 4. Save
df_clean.to_csv(OUTPUT_FILE, index=False)
print(f"Saved preprocessed data to {OUTPUT_FILE}")

Saved preprocessed data to data/stackoverflow_preprocessed.csv


XÓA CỘT ID, ĐƯA TIME VỀ ĐÚNG DẠNG